In [ ]:
# =========================================================
# Importing Necessary Libraries
# =========================================================

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

from sklearn.utils import class_weight

import os
import glob

from google.colab import drive

# =========================================================
# Mount Google Drive
# =========================================================

print("Mounting Google Drive...")
drive.mount('/content/drive')

# =========================================================
# 1. Data Loading and Splitting
# =========================================================

print("\n--- Section 1: Loading and Splitting Data ---")

BASE_DIR = '/content/drive/MyDrive/mini_project/chest_xray'

TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR = os.path.join(BASE_DIR, 'test')
VAL_DIR = os.path.join(BASE_DIR, 'val')

all_image_paths = []
all_labels = []

# Process all directories
for folder in [TRAIN_DIR, TEST_DIR, VAL_DIR]:

    normal_paths = glob.glob(
        os.path.join(folder, 'NORMAL', '*.jpeg')
    )

    pneumonia_paths = glob.glob(
        os.path.join(folder, 'PNEUMONIA', '*.jpeg')
    )

    all_image_paths.extend(normal_paths)
    all_labels.extend(['NORMAL'] * len(normal_paths))

    all_image_paths.extend(pneumonia_paths)
    all_labels.extend(['PNEUMONIA'] * len(pneumonia_paths))

# Create DataFrame
df = pd.DataFrame({
    'filepath': all_image_paths,
    'label': all_labels
})

# Shuffle dataset
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(f"Total images found: {len(df)}")
print(df['label'].value_counts())

# 70% Train | 10% Validation | 20% Test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,
    random_state=42,
    stratify=train_val_df['label']
)

print(f"\nTraining set:   {len(train_df)} images")
print(f"Validation set: {len(val_df)} images")
print(f"Test set:       {len(test_df)} images")

# =========================================================
# 2. Data Generators and Class Weights
# =========================================================

print("\n--- Section 2: Creating Data Generators ---")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
COLOR_MODE = 'rgb'

datagen = ImageDataGenerator(
    rescale=1./255
)

test_val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_generator = test_val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Test generator
test_generator = test_val_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label',
    target_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    batch_size=1,
    shuffle=False
)

# =========================================================
# Calculate Class Weights
# =========================================================

class_indices = train_generator.class_indices
labels = train_generator.classes

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights_dict = dict(
    zip(np.unique(labels), class_weights)
)

print(f"Class Indices: {class_indices}")
print(f"Class Weights: {class_weights_dict}")

# =========================================================
# 3. Model Definition and Callbacks
# =========================================================

print("\n--- Section 3: Defining Model and Callbacks ---")

INPUT_SHAPE = (224, 224, 3)

def build_proposed_model(dropout_rate=0.0):

    """
    Builds the custom CNN architecture.
    """

    img_input = Input(
        shape=INPUT_SHAPE,
        name='InputLayer'
    )

    # =====================================================
    # Block 1
    # =====================================================

    x = Conv2D(
        32,
        (3, 3),
        padding='same',
        name='Conv1_1'
    )(img_input)

    x = BatchNormalization(name='BN1_1')(x)
    x = Activation('relu', name='Act1_1')(x)

    x = Conv2D(
        32,
        (3, 3),
        padding='same',
        name='Conv1_2'
    )(x)

    x = BatchNormalization(name='BN1_2')(x)
    x = Activation('relu', name='Act1_2')(x)

    x = MaxPooling2D(
        (2, 2),
        strides=(2, 2),
        name='Pool1'
    )(x)

    # =====================================================
    # Block 2
    # =====================================================

    x = Conv2D(
        64,
        (3, 3),
        padding='same',
        name='Conv2_1'
    )(x)

    x = BatchNormalization(name='BN2_1')(x)
    x = Activation('relu', name='Act2_1')(x)

    x = Conv2D(
        64,
        (3, 3),
        padding='same',
        name='Conv2_2'
    )(x)

    x = BatchNormalization(name='BN2_2')(x)
    x = Activation('relu', name='Act2_2')(x)

    x = MaxPooling2D(
        (2, 2),
        strides=(2, 2),
        name='Pool2'
    )(x)

    # =====================================================
    # Block 3
    # =====================================================

    x = Conv2D(
        128,
        (3, 3),
        padding='same',
        name='Conv3_1'
    )(x)

    x = BatchNormalization(name='BN3_1')(x)
    x = Activation('relu', name='Act3_1')(x)

    x = Conv2D(
        128,
        (3, 3),
        padding='same',
        name='Conv3_2'
    )(x)

    x = BatchNormalization(name='BN3_2')(x)
    x = Activation('relu', name='Act3_2')(x)

    x = MaxPooling2D(
        (3, 3),
        strides=(3, 3),
        padding='same',
        name='Pool3'
    )(x)

    # =====================================================
    # Block 4
    # =====================================================

    x = Conv2D(
        256,
        (3, 3),
        padding='same',
        name='Conv4_1'
    )(x)

    x = BatchNormalization(name='BN4_1')(x)
    x = Activation('relu', name='Act4_1')(x)

    x = Conv2D(
        128,
        (3, 3),
        padding='same',
        name='Conv4_2'
    )(x)

    x = BatchNormalization(name='BN4_2')(x)
    x = Activation('relu', name='Act4_2')(x)

    x = MaxPooling2D(
        (3, 3),
        strides=(3, 3),
        padding='same',
        name='Pool4'
    )(x)

    # =====================================================
    # Block 5
    # =====================================================

    x = Conv2D(
        1024,
        (3, 3),
        padding='same',
        name='Conv5_1'
    )(x)

    x = BatchNormalization(name='BN5_1')(x)
    x = Activation('relu', name='Act5_1')(x)

    x = Conv2D(
        512,
        (3, 3),
        padding='same',
        name='Conv5_2'
    )(x)

    if dropout_rate > 0.0:
        x = Dropout(
            dropout_rate,
            name='Conv_Dropout'
        )(x)

    x = BatchNormalization(name='BN5_2')(x)
    x = Activation('relu', name='Act5_2')(x)

    x = MaxPooling2D(
        (3, 3),
        strides=(3, 3),
        padding='same',
        name='Pool5'
    )(x)

    # =====================================================
    # Dense Layers
    # =====================================================

    x = Flatten(name='Flatten')(x)

    x = Dense(512, name='Dense1')(x)
    x = Dense(1024, name='Dense2')(x)
    x = Dense(512, name='Dense3')(x)
    x = Dense(512, name='Dense4')(x)
    x = Dense(256, name='Dense5')(x)
    x = Dense(64, name='Dense6')(x)

    predictions = Dense(
        2,
        activation='softmax',
        name='predictions'
    )(x)

    model = Model(
        inputs=img_input,
        outputs=predictions
    )

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# =========================================================
# Callbacks
# =========================================================

reduce_lr = ReduceLROnPlateau(
    monitor='accuracy',
    factor=0.8,
    patience=5,
    cooldown=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

callbacks_list = [
    reduce_lr,
    early_stopping
]

EPOCHS = 50

# =========================================================
# Evaluation Function
# =========================================================

def evaluate_model(model, test_generator):

    test_generator.reset()

    y_pred_proba = model.predict(
        test_generator,
        steps=len(test_generator)
    )

    y_pred = np.argmax(
        y_pred_proba,
        axis=1
    )

    y_true = test_generator.classes

    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'F1_score': f1_score(y_true, y_pred),
        'AUC': roc_auc_score(
            y_true,
            y_pred_proba[:, 1]
        ),
        'y_true': y_true,
        'y_pred_proba': y_pred_proba
    }

    return metrics

# =========================================================
# Store Histories and Metrics
# =========================================================

model_histories = {}
model_metrics = {}

# =========================================================
# 4. Train Main Models
# =========================================================

print("\n--- Section 4: Training Main Models ---")

# =====================================================
# Model 1 : Without Dropout
# =====================================================

print("\nTraining Proposed Model (Without Dropout)...")

model_no_dropout = build_proposed_model(
    dropout_rate=0.0
)

history_no_dropout = model_no_dropout.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_list,
    class_weight=class_weights_dict,
    verbose=1
)

model_histories['Without dropout'] = history_no_dropout

model_metrics['Without dropout'] = evaluate_model(
    model_no_dropout,
    test_generator
)

# =====================================================
# Model 2 : With 40% Dropout
# =====================================================

print("\nTraining Proposed Model (With 40% Dropout)...")

model_with_dropout = build_proposed_model(
    dropout_rate=0.4
)

history_with_dropout = model_with_dropout.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_list,
    class_weight=class_weights_dict,
    verbose=1
)

model_histories['With dropout'] = history_with_dropout

model_metrics['With dropout'] = evaluate_model(
    model_with_dropout,
    test_generator
)